In [8]:
# Install required libraries
!pip install -q rank_bm25 sentence-transformers faiss-cpu polars pyarrow

In [11]:
from pathlib import Path

DATA_DIR = Path("/kaggle/input/datasets/shambhavigantla/mind-ebnerd-data1")

# Verify essential files exist
assert (DATA_DIR / "MINDsmall_train/MINDsmall_train/news.tsv").exists(), "MIND train news not found!"
assert (DATA_DIR / "ebnerd_demo/articles.parquet").exists(), "EB-NeRD demo articles not found!"

print("All dataset paths verified successfully!")

All dataset paths verified successfully!


In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/kaggle/input/datasets/shambhavigantla/mind-ebnerd-data1")

# 1. Parse MIND Small
def parse_mind(data_root):
    # Notice the nested folder path matching your directory tree
    train_news_p = data_root / "MINDsmall_train/MINDsmall_train/news.tsv"
    train_beh_p = data_root / "MINDsmall_train/MINDsmall_train/behaviors.tsv"
    
    news_cols = ["article_id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]
    news_df = pd.read_csv(train_news_p, sep="\t", header=None, names=news_cols, quoting=3)
    news_df["body"] = ""
    news_df["clean_text"] = news_df["title"].fillna("") + " " + news_df["abstract"].fillna("")

    beh_cols = ["impression_id", "user_id", "time", "history", "impressions"]
    beh_df = pd.read_csv(train_beh_p, sep="\t", header=None, names=beh_cols)
    beh_df["timestamp"] = pd.to_datetime(beh_df["time"])
    beh_df = beh_df.sort_values("timestamp").reset_index(drop=True)

    # Temporal split: 70% train, 15% val, 15% test
    n = len(beh_df)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    train_beh = beh_df.iloc[:train_end]
    val_beh = beh_df.iloc[train_end:val_end]
    test_beh = beh_df.iloc[val_end:]

    # Anti-gaming temporal assertion
    assert train_beh["timestamp"].max() <= val_beh["timestamp"].min() <= test_beh["timestamp"].min(), "Temporal boundary failure in MIND!"
    print(f"MIND parsed successfully: {len(news_df):,} articles, {len(train_beh):,} train impressions, {len(test_beh):,} test impressions.")

    return news_df, train_beh, val_beh, test_beh

# 2. Parse EB-NeRD Demo
def parse_ebnerd(data_root):
    eb_path = data_root / "ebnerd_demo"
    articles_df = pd.read_parquet(eb_path / "articles.parquet")
    articles_df = articles_df.rename(columns={"subtitle": "abstract"})
    articles_df["clean_text"] = articles_df["title"].fillna("") + " " + articles_df["abstract"].fillna("")

    train_beh = pd.read_parquet(eb_path / "train/behaviors.parquet")
    train_history = pd.read_parquet(eb_path / "train/history.parquet")
    
    # Merge user history
    train_beh = train_beh.merge(train_history, on="user_id", how="left")
    
    if "impression_time" in train_beh.columns:
        train_beh["timestamp"] = pd.to_datetime(train_beh["impression_time"])
    elif "time" in train_beh.columns:
        train_beh["timestamp"] = pd.to_datetime(train_beh["time"])
    else:
        train_beh["timestamp"] = pd.date_range(start="2024-01-01", periods=len(train_beh), freq="s")
        
    train_beh = train_beh.sort_values("timestamp").reset_index(drop=True)

    n = len(train_beh)
    train_split = train_beh.iloc[:int(n * 0.70)]
    val_split = train_beh.iloc[int(n * 0.70):int(n * 0.85)]
    test_split = train_beh.iloc[int(n * 0.85):]

    assert train_split["timestamp"].max() <= val_split["timestamp"].min() <= test_split["timestamp"].min(), "Temporal boundary failure in EB-NeRD!"
    print(f"EB-NeRD parsed successfully: {len(articles_df):,} articles, {len(train_split):,} train impressions, {len(test_split):,} test impressions.")

    return articles_df, train_split, val_split, test_split

# Execute parsing
mind_articles, mind_train, mind_val, mind_test = parse_mind(DATA_DIR)
eb_articles, eb_train, eb_val, eb_test = parse_ebnerd(DATA_DIR)

MIND parsed successfully: 51,282 articles, 109,875 train impressions, 23,545 test impressions.
EB-NeRD parsed successfully: 11,777 articles, 17,306 train impressions, 3,709 test impressions.


In [13]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/kaggle/input/datasets/shambhavigantla/mind-ebnerd-data1")

print("=" * 60)
print("1. INSPECTING MIND SMALL NEWS (First 2 rows)")
print("=" * 60)
mind_news_path = DATA_DIR / "MINDsmall_train/MINDsmall_train/news.tsv"
mind_news_sample = pd.read_csv(mind_news_path, sep="\t", header=None, nrows=2)
print("Shape / Columns count:", mind_news_sample.shape)
print(mind_news_sample.head())

print("\n" + "=" * 60)
print("2. INSPECTING MIND SMALL BEHAVIORS (First 2 rows)")
print("=" * 60)
mind_beh_path = DATA_DIR / "MINDsmall_train/MINDsmall_train/behaviors.tsv"
mind_beh_sample = pd.read_csv(mind_beh_path, sep="\t", header=None, nrows=2)
print("Shape / Columns count:", mind_beh_sample.shape)
print(mind_beh_sample.head())

print("\n" + "=" * 60)
print("3. INSPECTING EB-NeRD ARTICLES (First 2 rows)")
print("=" * 60)
eb_articles_path = DATA_DIR / "ebnerd_demo/articles.parquet"
eb_articles_sample = pd.read_parquet(eb_articles_path)
print("Columns:", eb_articles_sample.columns.tolist())
print(eb_articles_sample.head(2))

print("\n" + "=" * 60)
print("4. INSPECTING EB-NeRD TRAIN BEHAVIORS (First 2 rows)")
print("=" * 60)
eb_beh_path = DATA_DIR / "ebnerd_demo/train/behaviors.parquet"
eb_beh_sample = pd.read_parquet(eb_beh_path)
print("Columns:", eb_beh_sample.columns.tolist())
print(eb_beh_sample.head(2))

print("\n" + "=" * 60)
print("5. INSPECTING EB-NeRD TRAIN HISTORY (First 2 rows)")
print("=" * 60)
eb_hist_path = DATA_DIR / "ebnerd_demo/train/history.parquet"
eb_hist_sample = pd.read_parquet(eb_hist_path)
print("Columns:", eb_hist_sample.columns.tolist())
print(eb_hist_sample.head(2))

1. INSPECTING MIND SMALL NEWS (First 2 rows)
Shape / Columns count: (2, 8)
        0          1                2  \
0  N55528  lifestyle  lifestyleroyals   
1  N19639     health       weightloss   

                                                   3  \
0  The Brands Queen Elizabeth, Prince Charles, an...   
1                      50 Worst Habits For Belly Fat   

                                                   4  \
0  Shop the notebooks, jackets, and more that the...   
1  These seemingly harmless habits are holding yo...   

                                               5  \
0  https://assets.msn.com/labs/mind/AAGH0ET.html   
1  https://assets.msn.com/labs/mind/AAB19MK.html   

                                                   6  \
0  [{"Label": "Prince Philip, Duke of Edinburgh",...   
1  [{"Label": "Adipose tissue", "Type": "C", "Wik...   

                                                   7  
0                                                 []  
1  [{"Label": "Adipose tiss

In [17]:
!pip install -q rank_bm25

In [22]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# 1. Lexical Retrieval on MIND
# ==========================================
print("--- 1. Running Fast Vectorized Lexical Retrieval on MIND ---")

# Combine title and abstract
mind_articles["clean_text"] = mind_articles["title"].fillna("") + " " + mind_articles["abstract"].fillna("")

# Build Sparse Inverted Index via TF-IDF (Compiled C-level matrix multiplication)
tfidf_mind = TfidfVectorizer(stop_words='english', max_features=30000)
doc_matrix_mind = tfidf_mind.fit_transform(mind_articles["clean_text"])

mind_title_map = dict(zip(mind_articles["article_id"], mind_articles["title"]))
mind_idx_to_id = {idx: aid for idx, aid in enumerate(mind_articles["article_id"])}

mind_recalls = {50: [], 100: [], 200: []}
eval_count = 0

# Evaluate on first 300 test impressions for instant feedback
for _, row in mind_test.iterrows():
    if eval_count >= 300:
        break
        
    history = str(row["history"]).split()
    if not history or history == ['nan']:
        continue
        
    recent_titles = [str(mind_title_map[aid]) for aid in history[-5:] if aid in mind_title_map and pd.notna(mind_title_map[aid])]
    query_text = " ".join(recent_titles)
    if not query_text.strip():
        continue
        
    # Transform query to TF-IDF sparse vector & compute dot product
    query_vec = tfidf_mind.transform([query_text])
    scores = (query_vec * doc_matrix_mind.T).toarray()[0]
    
    # Top 200 items
    top_indices = np.argpartition(scores, -200)[-200:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    retrieved_ids = [mind_idx_to_id[i] for i in top_indices]
    
    # Extract ground truth clicks
    imp_parts = str(row["impressions"]).split()
    ground_truth = [p.split("-")[0] for p in imp_parts if p.endswith("-1")]
    if not ground_truth:
        continue
        
    for k in [50, 100, 200]:
        hits = len(set(ground_truth).intersection(set(retrieved_ids[:k])))
        mind_recalls[k].append(hits / len(ground_truth))
        
    eval_count += 1

print(f"MIND Lexical Results ({eval_count} test impressions):")
for k, v in mind_recalls.items():
    print(f"  Recall@{k}: {np.mean(v):.4f}")

# ==========================================
# 2. Lexical Retrieval on EB-NeRD Demo
# ==========================================
print("\n--- 2. Running Fast Vectorized Lexical Retrieval on EB-NeRD Demo ---")

eb_articles["clean_text"] = eb_articles["title"].fillna("") + " " + eb_articles["abstract"].fillna("")

tfidf_eb = TfidfVectorizer(max_features=30000)
doc_matrix_eb = tfidf_eb.fit_transform(eb_articles["clean_text"])

eb_title_map = dict(zip(eb_articles["article_id"], eb_articles["title"]))
eb_idx_to_id = {idx: aid for idx, aid in enumerate(eb_articles["article_id"])}

eb_recalls = {50: [], 100: [], 200: []}
eval_count = 0

for _, row in eb_test.iterrows():
    if eval_count >= 300:
        break
        
    raw_hist = row.get("article_id_fixed", None)
    if raw_hist is None or (isinstance(raw_hist, float) and np.isnan(raw_hist)):
        continue
    history = [int(x) for x in list(raw_hist)] if hasattr(raw_hist, '__iter__') else []
    if not history:
        continue
        
    recent_titles = [str(eb_title_map[aid]) for aid in history[-5:] if aid in eb_title_map and pd.notna(eb_title_map[aid])]
    query_text = " ".join(recent_titles)
    if not query_text.strip():
        continue
        
    query_vec = tfidf_eb.transform([query_text])
    scores = (query_vec * doc_matrix_eb.T).toarray()[0]
    
    top_indices = np.argpartition(scores, -200)[-200:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    retrieved_ids = [eb_idx_to_id[i] for i in top_indices]
    
    raw_clicks = row.get("article_ids_clicked", None)
    if raw_clicks is None or (isinstance(raw_clicks, float) and np.isnan(raw_clicks)):
        continue
    ground_truth = [int(x) for x in list(raw_clicks)] if hasattr(raw_clicks, '__iter__') else []
    if not ground_truth:
        continue
        
    for k in [50, 100, 200]:
        hits = len(set(ground_truth).intersection(set(retrieved_ids[:k])))
        eb_recalls[k].append(hits / len(ground_truth))
        
    eval_count += 1

print(f"EB-NeRD Demo Lexical Results ({eval_count} test impressions):")
for k, v in eb_recalls.items():
    print(f"  Recall@{k}: {np.mean(v):.4f}")

--- 1. Running Fast Vectorized Lexical Retrieval on MIND ---
MIND Lexical Results (300 test impressions):
  Recall@50: 0.0261
  Recall@100: 0.0361
  Recall@200: 0.0678

--- 2. Running Fast Vectorized Lexical Retrieval on EB-NeRD Demo ---
EB-NeRD Demo Lexical Results (300 test impressions):
  Recall@50: 0.0033
  Recall@100: 0.0133
  Recall@200: 0.0167


In [24]:
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 56.0 MB/s eta 0:00:00:00:0100:01


In [25]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# 1. Load Pretrained Sentence Transformer Model
print("Loading Embedding Model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# ==========================================
# 1. Semantic Retrieval on MIND Small
# ==========================================
print("\n--- 1. Running Semantic Retrieval on MIND ---")

mind_texts = mind_articles["clean_text"].tolist()
print(f"Generating embeddings for {len(mind_texts):,} MIND articles...")

# Encode all article texts with normalization (L2 norm = 1 enables inner product cosine search)
mind_embeddings = embed_model.encode(
    mind_texts, 
    batch_size=256, 
    show_progress_bar=True, 
    normalize_embeddings=True
).astype('float32')

# Build FAISS Index (IndexFlatIP computes exact inner products / cosine similarity)
dim_mind = mind_embeddings.shape[1]
faiss_mind = faiss.IndexFlatIP(dim_mind)
faiss_mind.add(mind_embeddings)

mind_id_to_idx = {aid: idx for idx, aid in enumerate(mind_articles["article_id"])}
mind_idx_to_id = {idx: aid for idx, aid in enumerate(mind_articles["article_id"])}

mind_dense_recalls = {50: [], 100: [], 200: []}
eval_count = 0

for _, row in mind_test.iterrows():
    if eval_count >= 300:
        break
        
    history = str(row["history"]).split()
    if not history or history == ['nan']:
        continue
        
    # Get vector indices for user's history
    hist_indices = [mind_id_to_idx[aid] for aid in history[-5:] if aid in mind_id_to_idx]
    if not hist_indices:
        continue
        
    # User representation = Mean vector of recent clicks (normalized)
    user_vector = np.mean(mind_embeddings[hist_indices], axis=0, keepdims=True)
    user_vector = user_vector / np.linalg.norm(user_vector, axis=1, keepdims=True)
    
    # FAISS ANN / Dense Search
    _, top_indices = faiss_mind.search(user_vector.astype('float32'), 200)
    retrieved_ids = [mind_idx_to_id[i] for i in top_indices[0]]
    
    # Ground truth clicks
    imp_parts = str(row["impressions"]).split()
    ground_truth = [p.split("-")[0] for p in imp_parts if p.endswith("-1")]
    if not ground_truth:
        continue
        
    for k in [50, 100, 200]:
        hits = len(set(ground_truth).intersection(set(retrieved_ids[:k])))
        mind_dense_recalls[k].append(hits / len(ground_truth))
        
    eval_count += 1

print(f"MIND Dense Embedding Results ({eval_count} test impressions):")
for k, v in mind_dense_recalls.items():
    print(f"  Recall@{k}: {np.mean(v):.4f}")

# ==========================================
# 2. Semantic Retrieval on EB-NeRD Demo
# ==========================================
print("\n--- 2. Running Semantic Retrieval on EB-NeRD Demo ---")

eb_texts = eb_articles["clean_text"].tolist()
print(f"Generating embeddings for {len(eb_texts):,} EB-NeRD articles...")

eb_embeddings = embed_model.encode(
    eb_texts, 
    batch_size=256, 
    show_progress_bar=True, 
    normalize_embeddings=True
).astype('float32')

dim_eb = eb_embeddings.shape[1]
faiss_eb = faiss.IndexFlatIP(dim_eb)
faiss_eb.add(eb_embeddings)

eb_id_to_idx = {str(aid): idx for idx, aid in enumerate(eb_articles["article_id"])}
eb_idx_to_id = {idx: aid for idx, aid in enumerate(eb_articles["article_id"])}

eb_dense_recalls = {50: [], 100: [], 200: []}
eval_count = 0

for _, row in eb_test.iterrows():
    if eval_count >= 300:
        break
        
    raw_hist = row.get("article_id_fixed", None)
    if raw_hist is None or (isinstance(raw_hist, float) and np.isnan(raw_hist)):
        continue
    history = [str(x) for x in list(raw_hist)] if hasattr(raw_hist, '__iter__') else []
    if not history:
        continue
        
    hist_indices = [eb_id_to_idx[aid] for aid in history[-5:] if aid in eb_id_to_idx]
    if not hist_indices:
        continue
        
    user_vector = np.mean(eb_embeddings[hist_indices], axis=0, keepdims=True)
    user_vector = user_vector / np.linalg.norm(user_vector, axis=1, keepdims=True)
    
    _, top_indices = faiss_eb.search(user_vector.astype('float32'), 200)
    retrieved_ids = [eb_idx_to_id[i] for i in top_indices[0]]
    
    raw_clicks = row.get("article_ids_clicked", None)
    if raw_clicks is None or (isinstance(raw_clicks, float) and np.isnan(raw_clicks)):
        continue
    ground_truth = [int(x) for x in list(raw_clicks)] if hasattr(raw_clicks, '__iter__') else []
    if not ground_truth:
        continue
        
    for k in [50, 100, 200]:
        hits = len(set(ground_truth).intersection(set(retrieved_ids[:k])))
        eb_dense_recalls[k].append(hits / len(ground_truth))
        
    eval_count += 1

print(f"EB-NeRD Dense Embedding Results ({eval_count} test impressions):")
for k, v in eb_dense_recalls.items():
    print(f"  Recall@{k}: {np.mean(v):.4f}")

Loading Embedding Model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...


--- 1. Running Semantic Retrieval on MIND ---
Generating embeddings for 51,282 MIND articles...


Batches:   0%|          | 0/201 [00:00<?, ?it/s]

MIND Dense Embedding Results (300 test impressions):
  Recall@50: 0.0133
  Recall@100: 0.0200
  Recall@200: 0.0396

--- 2. Running Semantic Retrieval on EB-NeRD Demo ---
Generating embeddings for 11,777 EB-NeRD articles...


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

EB-NeRD Dense Embedding Results (300 test impressions):
  Recall@50: 0.0033
  Recall@100: 0.0033
  Recall@200: 0.0067
